In [1]:
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate # for string prompt
from langchain_core.output_parsers import PydanticOutputParser

PydanticOutputParser -> to get the structure 

Field -> To tag the data (e.g. price, name)

BaseModel -> to create structure output for pydantic model

In [2]:
with open('data.txt','r+') as file:
    data = file.read()

In [3]:
# model = ChatOllama(
#     model="qwen2.5:1.5b", 
#     temperature=0
#     )
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
    )

In [4]:
template = '''
You are a helpful assistant that can answer questions about the product.
Answer the question based on the product details provided.
product : {product}
question : {question}
'''

prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['product', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant that can answer questions about the product.\nAnswer the question based on the product details provided.\nproduct : {product}\nquestion : {question}\n')

In [5]:
class Product(BaseModel):
    name:str = Field(description="Name of the product")
    price:str  = Field(description="Price of the product")
    description:str = Field(description="Description of the product")
    category:str = Field(description="Category of the product")
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
    ).with_structured_output(Product)

In [6]:
chain = prompt_template | model

In [7]:
user_input = [
    {
    "product": data,
    "question": "What is the price of the iPhone 15 Pro Max?"
},
{
    "product": data,
    "question": "What is the price of the Galaxy S24 Ultra?"
},
{
    "product": data,
    "question": "What is the price of the Redmi?"
},
{
    "product": data,
    "question": "What is the price of the OnePlus?"
}
]

In [8]:
prompt_template = PromptTemplate.from_template(template)
chain = prompt_template | model 

In [9]:
u_input = {"product":data,'question':"What is the price of One plus 13?"}
response = chain.invoke(u_input)
response_dict = response.model_dump_json()
print(response_dict)

{"name":"OnePlus 12R","price":"45999.00","description":"OnePlus 12R with 16GB RAM, 256GB storage, 5500mAh battery, 6.78 inch AMOLED display, and Snapdragon 8 Gen 2 processor.","category":"Mobile"}


In [10]:
print(response)

name='OnePlus 12R' price='45999.00' description='OnePlus 12R with 16GB RAM, 256GB storage, 5500mAh battery, 6.78 inch AMOLED display, and Snapdragon 8 Gen 2 processor.' category='Mobile'


In [11]:
response

Product(name='OnePlus 12R', price='45999.00', description='OnePlus 12R with 16GB RAM, 256GB storage, 5500mAh battery, 6.78 inch AMOLED display, and Snapdragon 8 Gen 2 processor.', category='Mobile')

In [12]:
batch_response = chain.batch(user_input)

In [13]:
results = []
for result in  batch_response:
    results.append(result.model_dump_json())
results

['{"name":"Apple iPhone 15 Pro Max","price":"159999.00","description":"Apple iPhone 15 Pro Max with 8GB RAM and 256GB storage.","category":"Mobile Phones"}',
 '{"name":"Samsung Galaxy S24 Ultra","price":"129999.99","description":"Samsung Galaxy S24 Ultra with 12GB RAM, 512GB storage, 5000mAh battery, 6.8 inch AMOLED display, and Snapdragon 8 Gen 3 processor.","category":"Mobile Phones"}',
 '{"name":"Redmi Note 13 Pro+","price":"29999.99","description":"Redmi Note 13 Pro+ with 8GB RAM, 256GB storage, and Dimensity 7200 Ultra processor.","category":"Mobile Phones"}',
 '{"name":"OnePlus 12R","price":"45999.00","description":"OnePlus 12R with 16GB RAM, 256GB storage, 5500mAh battery, 6.78 inch AMOLED display, and Snapdragon 8 Gen 2 processor.","category":"Mobile Phones"}']

In [14]:
batch_response

[Product(name='Apple iPhone 15 Pro Max', price='159999.00', description='Apple iPhone 15 Pro Max with 8GB RAM and 256GB storage.', category='Mobile Phones'),
 Product(name='Samsung Galaxy S24 Ultra', price='129999.99', description='Samsung Galaxy S24 Ultra with 12GB RAM, 512GB storage, 5000mAh battery, 6.8 inch AMOLED display, and Snapdragon 8 Gen 3 processor.', category='Mobile Phones'),
 Product(name='Redmi Note 13 Pro+', price='29999.99', description='Redmi Note 13 Pro+ with 8GB RAM, 256GB storage, and Dimensity 7200 Ultra processor.', category='Mobile Phones'),
 Product(name='OnePlus 12R', price='45999.00', description='OnePlus 12R with 16GB RAM, 256GB storage, 5500mAh battery, 6.78 inch AMOLED display, and Snapdragon 8 Gen 2 processor.', category='Mobile Phones')]

In [15]:
for prompt in user_input:
    response = chain.invoke(prompt)
    print(response.model_dump())

{'name': 'Apple iPhone 15 Pro Max', 'price': '159999.00', 'description': 'Apple smartphone with 8GB RAM and 256GB storage, featuring A17 Pro processor and 6.7 inch Super Retina XDR display.', 'category': 'Mobile Phones'}
{'name': 'Samsung Galaxy S24 Ultra', 'price': '129999.99', 'description': 'Samsung Galaxy S24 Ultra with 12GB RAM, 512GB storage, 5000mAh battery, 6.8 inch AMOLED display, and Snapdragon 8 Gen 3 processor.', 'category': 'Mobile'}
{'name': 'Redmi Note 13 Pro+', 'price': '29999.99', 'description': 'Redmi smartphone with 8GB RAM, 256GB storage, 5000mAh battery, 6.67 inch AMOLED display, and Dimensity 7200 Ultra processor.', 'category': 'Mobile'}
{'name': 'OnePlus 12R', 'price': '45999.00', 'description': 'OnePlus 12R with 16GB RAM, 256GB storage, and Snapdragon 8 Gen 2 processor.', 'category': 'Mobile'}
